# VAR Hands-on Übung: Vector Autoregression von Grund auf

**Dauer: ca. 20 Minuten · Format: Selbststudium / Live im Kurs**

In dieser Übung baust du ein **Vector-Autoregression-Modell (VAR)** Schritt für
Schritt selbst nach – genau die Logik, die hinter dem Live-Demo-Dashboard steckt.
Du nutzt nur `numpy` und `pandas`, damit jeder Rechenschritt sichtbar bleibt.

## Datensatz
US-Makrodaten der **FRED** (Monatsdaten): Inflation, Arbeitslosigkeit,
Fed Funds Rate und Industrieproduktion. Leitfrage:
> *Wie hängen diese vier Größen über die Zeit zusammen, und lässt sich daraus
> eine bessere Prognose bauen als mit einer naiven Benchmark?*

## So gehst du vor
1. Führe die vorbereiteten Zellen (Daten, Plots) der Reihe nach aus.
2. Bearbeite die mit **`# TODO`** markierten Stellen. Jede ist nur wenige Zeilen.
3. Nach jeder Aufgabe gibt es eine **Self-Check-Zelle**. Erscheint `OK`, passt es.

## Lernziele
- Verstehen, wie ein VAR(p) als Regressionsproblem aufgebaut ist.
- Lag-Matrix, OLS-Schätzung, Lag-Auswahl (AIC/BIC), Mehrschritt-Forecast und
  Stabilität selbst implementieren.
- Den VAR-Forecast gegen eine naive Benchmark bewerten.

> Ein ausführlich kommentiertes **Lösungs-Notebook** gibt es im Anschluss.


## 0 · Setup und Daten laden

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.float_format", lambda v: f"{v:0.3f}")
np.set_printoptions(suppress=True, precision=3)

# Datenpfad robust finden (egal ob das Notebook im Projektordner oder in einem
# Unterordner liegt).
CANDIDATES = [Path("data"), Path("../data"), Path("../../data")]
DATA_DIR = next((p for p in CANDIDATES if (p / "var_fred_macro.csv").exists()), None)
assert DATA_DIR is not None, (
    "var_fred_macro.csv wurde nicht gefunden. "
    "Lege das Notebook in der Naehe des data-Ordners ab."
)

COLS = [
    "inflation_yoy",
    "unemployment_rate",
    "fed_funds_rate",
    "industrial_production_growth_yoy",
]
NAMES = {
    "inflation_yoy": "Inflation",
    "unemployment_rate": "Arbeitslosigkeit",
    "fed_funds_rate": "Fed Funds Rate",
    "industrial_production_growth_yoy": "Industrieproduktion",
}

data = (
    pd.read_csv(DATA_DIR / "var_fred_macro.csv", parse_dates=["date"])
    .set_index("date")[COLS]
    .dropna()
)
print("Form (Zeilen, Spalten):", data.shape)
print("Zeitraum:", data.index.min().date(), "bis", data.index.max().date())
data.head()

In [ ]:
def check(name, condition):
    print(("OK   " if condition else "FALSCH ") + name)
    assert condition, f"Self-Check fehlgeschlagen: {name}"


Ein erster Blick auf die Reihen – Ausreißer, Trends und Skalen sieht man hier schneller als in Koeffizienten:

In [ ]:
fig, axes = plt.subplots(len(COLS), 1, figsize=(10, 8), sharex=True)
for ax, c in zip(axes, COLS):
    ax.plot(data.index, data[c], lw=1.2)
    ax.set_title(NAMES[c], loc="left", fontsize=10)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fuer das Verstaendnis des Modells arbeiten wir zunaechst auf z-standardisierten
# Reihen (Mittelwert 0, Standardabweichung 1). Das macht Koeffizienten und
# Impulsantworten ueber unterschiedliche Skalen hinweg vergleichbar.
# Hinweis: Fuer die spaetere FAIRE Forecast-Bewertung standardisieren wir noch
# einmal getrennt - nur mit den Trainingsdaten (kein Data-Leakage).
Z = (data - data.mean()) / data.std(ddof=0)
Z.describe().T[["mean", "std", "min", "max"]]

## Kurzer Theorie-Spickzettel

Ein **VAR(p)** erklärt einen Vektor $y_t$ aus seinen eigenen Vergangenheitswerten
**und** denen der anderen Variablen:

$$ y_t = c + A_1 y_{t-1} + A_2 y_{t-2} + \dots + A_p y_{t-p} + u_t $$

Praktisch ist das eine **lineare Regression**: Wir stapeln je Zeitpunkt einen
Intercept und die letzten $p$ beobachteten Vektoren in eine Designmatrix $X$ und
schätzen die Koeffizienten per OLS. Genau das bauen wir jetzt.


## TODO 1 · Lag-Matrix bauen

Vervollständige `make_lag_matrix`. Ziel: aus dem DataFrame `frame` eine
Designmatrix `X = [const | y_{t-1} | ... | y_{t-p}]` und die Zielmatrix `Y = y_t`
machen. **Zwei Lücken** sind zu füllen.


In [ ]:
def make_lag_matrix(frame, p):
    # Baut die Designmatrix X und die Zielmatrix Y fuer ein VAR(p).
    # X = [const | y_{t-1} | y_{t-2} | ... | y_{t-p}],  Y = y_t
    arr = np.asarray(frame, dtype=float)
    n, k = arr.shape

    X_parts = [np.ones((n - p, 1))]      # 1. Spalte: Intercept (Einsen)
    names = ["const"]

    for lag in range(1, p + 1):
        # TODO 1a: Haenge den um `lag` verschobenen Block an X_parts an.
        #          Die passenden Zeilen sind arr[p - lag : n - lag].
        X_parts.append(...)  # <-- ersetzen
        names += [f"{c}_lag{lag}" for c in frame.columns]

    X = np.hstack(X_parts)
    # TODO 1b: Y sind die "heutigen" Werte ab Zeile p, also arr[p:].
    Y = ...  # <-- ersetzen
    return X, Y, names

In [ ]:
X, Y, names = make_lag_matrix(Z, 2)
k = Z.shape[1]
check("X hat Form (n-p, 1 + k*p)", X.shape == (len(Z) - 2, 1 + k * 2))
check("Y hat Form (n-p, k)", Y.shape == (len(Z) - 2, k))
check("erste Spalte ist Intercept (alles 1)", np.allclose(X[:, 0], 1.0))
check("Lag-1-Block ist korrekt verschoben", np.allclose(X[:, 1:1 + k], Z.values[1:-1]))
check("Y entspricht arr[p:]", np.allclose(Y, Z.values[2:]))
print("Spaltennamen:", names)

## TODO 2 · VAR per OLS schätzen

`np.linalg.lstsq(X, Y, rcond=None)[0]` löst alle Gleichungen gemeinsam. Fülle die
Koeffizienten- und die Residuen-Zeile aus.


In [ ]:
def fit_var(frame, p):
    X, Y, names = make_lag_matrix(frame, p)

    # TODO 2a: Schaetze die Koeffizienten per Kleinste-Quadrate (OLS).
    #          np.linalg.lstsq(X, Y, rcond=None)[0] loest alle k Gleichungen
    #          gleichzeitig und liefert eine Matrix der Form (Spalten von X, k).
    coef = ...  # <-- ersetzen

    # TODO 2b: Residuen = tatsaechliche minus vorhergesagte Werte.
    resid = ...  # <-- ersetzen

    sigma_u = (resid.T @ resid) / len(resid)   # Residual-Kovarianzmatrix
    return {
        "p": p,
        "columns": list(frame.columns),
        "coef": coef,
        "resid": resid,
        "sigma_u": sigma_u,
        "x_names": names,
    }

In [ ]:
m = fit_var(Z, 2)
X, Y, _ = make_lag_matrix(Z, 2)
check("coef hat Form (1 + k*p, k)", m["coef"].shape == (1 + Z.shape[1] * 2, Z.shape[1]))
check("Residuen stehen senkrecht auf X (OLS-Eigenschaft)", np.allclose(X.T @ m["resid"], 0, atol=1e-6))
check("sigma_u ist k x k", m["sigma_u"].shape == (Z.shape[1], Z.shape[1]))

# Koeffizienten von Lag 1 als kleine Heatmap-Tabelle (Zeile = Ziel heute,
# Spalte = Quelle in t-1).
k = Z.shape[1]
block = pd.DataFrame(m["coef"][1:1 + k, :].T, index=COLS, columns=COLS)
block.rename(index=NAMES, columns=NAMES).round(3)

## TODO 3 · Lag-Ordnung mit AIC/BIC wählen

Beide Kriterien belohnen kleine Residuen, bestrafen aber viele Parameter.
**Kleiner ist besser.** Trage die beiden Formeln ein.


In [ ]:
def information_criteria(frame, max_lag):
    k = frame.shape[1]
    rows = []
    for p in range(1, max_lag + 1):
        m = fit_var(frame, p)
        n = len(m["resid"])
        # log-Determinante der Residual-Kovarianz (numerisch stabil):
        _, logdet = np.linalg.slogdet(m["sigma_u"])
        params = k * (k * p + 1)               # Anzahl Parameter im System

        # TODO 3: AIC und BIC. Kleiner ist besser.
        #   aic = logdet + 2 * params / n
        #   bic = logdet + log(n) * params / n
        aic = ...  # <-- ersetzen
        bic = ...  # <-- ersetzen

        rows.append({"p": p, "aic": aic, "bic": bic})
    return pd.DataFrame(rows)

In [ ]:
crit = information_criteria(Z, 8)
check("Spalten p, aic, bic vorhanden", {"p", "aic", "bic"}.issubset(crit.columns))
check("alle Kriterien endlich", np.isfinite(crit[["aic", "bic"]].values).all())

p_aic = int(crit.loc[crit["aic"].idxmin(), "p"])
p_bic = int(crit.loc[crit["bic"].idxmin(), "p"])
print(f"AIC waehlt p = {p_aic},  BIC waehlt p = {p_bic}")
crit.set_index("p").round(4)

## TODO 4 · Mehrschritt-Forecast

Beim echten Mehrschritt-Forecast wird jede Prognose wieder als „Vergangenheit“
eingespeist. Fülle die Vorhersage-Zeile und das Anhängen an die Historie aus.


In [ ]:
def forecast_var(model, history, steps):
    # Echter Mehrschritt-Forecast: vorhergesagte Werte werden Schritt fuer
    # Schritt wieder als "Vergangenheit" eingespeist.
    p = int(model["p"])
    coef = model["coef"]
    hist = [row.copy() for row in np.asarray(history, dtype=float)[-p:]]
    preds = []

    for _ in range(steps):
        x = [1.0]                              # Intercept
        for lag in range(1, p + 1):
            x.extend(hist[-lag])               # zuletzt bekannte Werte anhaengen

        # TODO 4a: ein Schritt vorhersagen: y = x (als array) @ coef
        y = ...  # <-- ersetzen

        preds.append(y)
        # TODO 4b: y an die Historie anhaengen, damit der naechste Schritt
        #          darauf aufbauen kann.
        ...  # <-- ersetzen (hist.append(...))

    return np.asarray(preds)

In [ ]:
m = fit_var(Z, 2)
f = forecast_var(m, Z, 5)
check("Forecast hat Form (steps, k)", f.shape == (5, Z.shape[1]))

# Der erste Schritt muss exakt x_last @ coef sein.
x_last = np.concatenate([[1.0], Z.values[-1], Z.values[-2]])
check("1-Schritt-Forecast ist konsistent", np.allclose(f[0], x_last @ m["coef"]))
print("Erste 3 prognostizierte (z-)Werte:")
pd.DataFrame(f, columns=COLS).head(3).rename(columns=NAMES).round(3)

## TODO 5 · VAR gegen naive Benchmark testen

Die nächste Zelle ist **vorbereitet**: sauberer Train/Test-Split, Standardisierung
nur auf Trainingsdaten (kein Leakage), Lag per BIC, Forecast und Rückskalierung.
Führe sie aus – danach ist nur der RMSE-Vergleich deine Aufgabe.


In [ ]:
# Train/Test-Split nach Zeit: die letzten H Monate sind Test.
H = 24
train = data.iloc[:-H]
test = data.iloc[-H:]

# Standardisieren NUR mit Trainings-Statistik -> kein Data-Leakage.
mu = train.mean()
sigma = train.std(ddof=0)
train_z = (train - mu) / sigma

# Lag per BIC auf den Trainingsdaten waehlen.
crit_train = information_criteria(train_z, 12)
p = int(crit_train.loc[crit_train["bic"].idxmin(), "p"])
print("Gewaehlter Lag p (BIC):", p)

# Modell schaetzen und H Schritte prognostizieren, danach zurueckskalieren.
model = fit_var(train_z, p)
fc_z = forecast_var(model, train_z, H)
fc = pd.DataFrame(fc_z * sigma.values + mu.values, index=test.index, columns=COLS)

# Naive Benchmark: letzter bekannter Wert wird einfach fortgeschrieben.
naive = pd.DataFrame(
    np.repeat(train.iloc[[-1]].values, H, axis=0), index=test.index, columns=COLS
)
fc.head()

In [ ]:
# TODO 5: RMSE (Wurzel des mittleren quadratischen Fehlers) je Variable.
#   rmse = sqrt( mean( (prognose - ist)^2 ) )  -> mit pandas spaltenweise.
rmse_var = ...    # <-- ersetzen: VAR-Forecast `fc` gegen `test`
rmse_naive = ...  # <-- ersetzen: `naive` gegen `test`

vergleich = pd.DataFrame({"VAR": rmse_var, "Naiv": rmse_naive})
vergleich["Gewinner"] = np.where(vergleich["VAR"] < vergleich["Naiv"], "VAR", "Naiv")
vergleich.rename(index=NAMES).round(3)

Und der visuelle Vergleich für eine Zielvariable:

In [ ]:
ziel = "inflation_yoy"   # gern aendern: eine der vier Variablen
plt.figure(figsize=(11, 4))
plt.plot(train.index[-60:], train[ziel].iloc[-60:], label="Train Ist", color="black")
plt.plot(test.index, test[ziel], label="Test Ist", color="tab:blue", lw=2)
plt.plot(fc.index, fc[ziel], "--", label="VAR Forecast", color="tab:red")
plt.plot(naive.index, naive[ziel], ":", label="Naiv", color="gray")
plt.axvline(test.index[0], color="red", lw=1, alpha=0.5)
plt.title(f"Forecast-Test: {NAMES[ziel]}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Bonus · Stabilität prüfen

Ist das Modell stabil? Gib die Eigenwerte der Companion-Matrix zurück. Liegt der
größte Betrag unter 1, klingen Schocks ab und Prognosen explodieren nicht.


In [ ]:
def companion_roots(model):
    # Stabilitaet: ein VAR ist stabil, wenn alle Eigenwerte der Companion-Matrix
    # betragsmaessig < 1 sind (Schocks klingen ab, Prognosen explodieren nicht).
    k = len(model["columns"])
    p = int(model["p"])
    coef = model["coef"]

    # Obere Bloecke: die transponierten Lag-Koeffizientenmatrizen nebeneinander.
    blocks = [coef[1 + lag * k : 1 + (lag + 1) * k, :].T for lag in range(p)]
    top = np.hstack(blocks)

    if p == 1:
        companion = top
    else:
        lower = np.hstack([np.eye(k * (p - 1)), np.zeros((k * (p - 1), k))])
        companion = np.vstack([top, lower])

    # TODO Bonus: Eigenwerte der Companion-Matrix zurueckgeben.
    return ...  # <-- ersetzen: np.linalg.eigvals(companion)


roots = companion_roots(fit_var(Z, p))
print("groesster Betrag der Wurzeln:", np.max(np.abs(roots)).round(3))
print("stabil?", bool(np.max(np.abs(roots)) < 1))

## Reflexion (kurz für dich notieren)

1. **AIC vs. BIC:** Welches Kriterium wählt mehr Lags – und warum ist das so?
2. **Benchmark:** Bei welchen Variablen schlägt das VAR die naive Prognose,
   bei welchen nicht? Was könnte der Grund sein?
3. **Leakage:** Warum standardisieren wir mit den *Trainings*-Statistiken und
   nicht mit dem gesamten Datensatz?
4. **Kausalität:** Darf man aus einem VAR-Koeffizienten oder einer Impulsantwort
   direkt „A verursacht B“ schließen? Warum (nicht)?

> ✅ Fertig? Vergleiche deine Lösung mit dem **Lösungs-Notebook**
> (`VAR_Uebung_Loesung.ipynb`). Dort findest du ausführliche Erklärungen,
> Impulsantworten und eine Einordnung der Ergebnisse.
